# Notebook 05: Gold Budget and Plan fact

## Runtime contract

- **Inputs:** current-state Silver `budget` and `faglflexp` tables.
- **Output grain:** plan source, scenario, company, account, cost center, profit center, fiscal year, fiscal period, and plan version.
- **Transformation:** long-form budget rows are unioned with FAGLFLEXP periods `TSL01` through `TSL16` and resolved to current planning dimensions.
- **Replay behavior:** an idempotent Delta MERGE updates or inserts by `BUDGET_FACT_SK`.
- **Failure evidence:** reconciliation by plan source, scenario, and version appends to `audit.budget_fact_reconciliation`.

In [ ]:
from pyspark.sql import functions as F, types as T, Window
from delta.tables import DeltaTable
from functools import reduce
import datetime
import uuid

BRZ_SLV_LH = "lh_brz_slv_finance"
GLD_LH = "lh_gld_finance"
SLV = f"{BRZ_SLV_LH}.slv"
AUD = f"{BRZ_SLV_LH}.audit"
DIM = f"{GLD_LH}.dim"
FCT = f"{GLD_LH}.fct"
HIGH_DATE = "9999-12-31"
BATCH_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)

In [ ]:
# Silver stores one current row per business key after full-snapshot MERGE.
BUDGET_KEYS = [
    "COMPANY_CODE", "GL_ACCOUNT", "COST_CENTER", "PROFIT_CENTER",
    "FISCAL_YEAR", "FISCAL_PERIOD", "PLAN_VERSION",
]
PLAN_KEYS = [
    "RCLNT", "RLDNR", "RBUKRS", "RYEAR", "RACCT", "PRCTR", "VERSN",
]

budget = spark.table(f"{SLV}.budget")
budget_long = budget.select(
    F.col("COMPANY_CODE"),
    F.col("GL_ACCOUNT"),
    F.col("COST_CENTER"),
    F.col("PROFIT_CENTER"),
    F.col("FISCAL_YEAR"),
    F.col("FISCAL_PERIOD"),
    F.coalesce(F.col("PLAN_VERSION"), F.lit("BUDGET")).alias("PLAN_VERSION"),
    F.lit("BUDGET_FILE").alias("PLAN_SOURCE"),
    F.lit("BUDGET").alias("SCENARIO"),
    F.col("BUDGET_AMOUNT_LOCAL")
    .cast("decimal(23,2)")
    .alias("PLAN_AMOUNT_LOCAL"),
    F.col("FORECAST_AMOUNT_LOCAL")
    .cast("decimal(23,2)")
    .alias("FORECAST_AMOUNT_LOCAL"),
    "BATCH_RUN_ID",
    "EXTRACT_DATE",
    "LOAD_DATE",
    "SOURCE_SYSTEM",
    "SOURCE_FILE_NAME",
)

# FAGLFLEXP stores periods as columns; explode preserves one row per plan period.
plan = spark.table(f"{SLV}.faglflexp")
period_structs = [
    F.struct(
        F.lit(period).alias("FISCAL_PERIOD"),
        F.col(f"TSL{period:02d}")
        .cast("decimal(23,2)")
        .alias("PLAN_AMOUNT_LOCAL"),
    )
    for period in range(1, 17)
]
plan_long = (
    plan.withColumn("_PERIOD", F.explode(F.array(*period_structs)))
    .select(
        F.col("RBUKRS").alias("COMPANY_CODE"),
        F.col("RACCT").alias("GL_ACCOUNT"),
        F.lit(None).cast("string").alias("COST_CENTER"),
        F.col("PRCTR").alias("PROFIT_CENTER"),
        F.col("RYEAR").alias("FISCAL_YEAR"),
        F.col("_PERIOD.FISCAL_PERIOD").alias("FISCAL_PERIOD"),
        F.col("VERSN").alias("PLAN_VERSION"),
        F.lit("FAGLFLEXP").alias("PLAN_SOURCE"),
        F.lit("PLAN").alias("SCENARIO"),
        F.col("_PERIOD.PLAN_AMOUNT_LOCAL").alias("PLAN_AMOUNT_LOCAL"),
        F.lit(None).cast("decimal(23,2)").alias("FORECAST_AMOUNT_LOCAL"),
        "BATCH_RUN_ID",
        "EXTRACT_DATE",
        "LOAD_DATE",
        "SOURCE_SYSTEM",
        "SOURCE_FILE_NAME",
    )
)

staged = budget_long.unionByName(plan_long)

# Forward-looking records intentionally resolve against current dimension versions.
co = spark.table(f"{DIM}.company").filter(F.col("IS_CURRENT") == True).alias("co")
gl = spark.table(f"{DIM}.gl_account").filter(F.col("IS_CURRENT") == True).alias("gl")
cc = spark.table(f"{DIM}.cost_center").filter(F.col("IS_CURRENT") == True).alias("cc")
pc = spark.table(f"{DIM}.profit_center").filter(F.col("IS_CURRENT") == True).alias("pc")
dv = spark.table(f"{DIM}.division").filter(F.col("IS_CURRENT") == True).alias("dv")

resolved = (
    staged.alias("s")
    .join(
        co,
        F.col("s.COMPANY_CODE").eqNullSafe(F.col("co.BUKRS")),
        "left",
    )
    .join(
        gl,
        F.col("s.GL_ACCOUNT").eqNullSafe(F.col("gl.SAKNR"))
        & F.col("co.KTOPL").eqNullSafe(F.col("gl.KTOPL")),
        "left",
    )
    .join(
        cc,
        F.col("s.COST_CENTER").eqNullSafe(F.col("cc.KOSTL")),
        "left",
    )
    .join(
        pc,
        F.col("s.PROFIT_CENTER").eqNullSafe(F.col("pc.PRCTR")),
        "left",
    )
    .join(
        dv,
        F.col("pc.KHINR").eqNullSafe(F.col("dv.DIVISION_CODE")),
        "left",
    )
)

grain = [
    "PLAN_SOURCE", "SCENARIO", "COMPANY_CODE", "GL_ACCOUNT",
    "COST_CENTER", "PROFIT_CENTER", "FISCAL_YEAR", "FISCAL_PERIOD",
    "PLAN_VERSION",
]
grain_values = [
    F.coalesce(F.col(f"s.{column_name}").cast("string"), F.lit("__NULL__"))
    for column_name in grain
]
out = resolved.select(
    F.sha2(F.concat_ws("||", *grain_values), 256).alias("BUDGET_FACT_SK"),
    ((F.col("s.FISCAL_YEAR") * F.lit(100)) + F.col("s.FISCAL_PERIOD"))
    .cast("int")
    .alias("FISCAL_PERIOD_KEY"),
    F.col("co.DIMENSION_SK").alias("COMPANY_SK"),
    F.col("gl.DIMENSION_SK").alias("GL_ACCOUNT_SK"),
    F.col("cc.DIMENSION_SK").alias("COST_CENTER_SK"),
    F.col("pc.DIMENSION_SK").alias("PROFIT_CENTER_SK"),
    F.col("dv.DIMENSION_SK").alias("DIVISION_SK"),
    *[F.col(f"s.{column_name}").alias(column_name) for column_name in grain],
    F.col("s.PLAN_AMOUNT_LOCAL"),
    F.col("s.FORECAST_AMOUNT_LOCAL"),
    F.col("s.BATCH_RUN_ID").alias("SOURCE_BATCH_RUN_ID"),
    F.col("s.EXTRACT_DATE"),
    F.col("s.LOAD_DATE").alias("SILVER_LOAD_DATE"),
    F.col("s.SOURCE_SYSTEM"),
    F.col("s.SOURCE_FILE_NAME"),
    F.lit(BATCH_ID).alias("GOLD_BATCH_ID"),
    F.current_timestamp().alias("GOLD_LOAD_TIMESTAMP"),
)

target = f"{FCT}.budget"
if spark.catalog.tableExists(target):
    (
        DeltaTable.forName(spark, target)
        .alias("tgt")
        .merge(out.alias("src"), "tgt.BUDGET_FACT_SK = src.BUDGET_FACT_SK")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    out.write.format("delta").mode("overwrite").saveAsTable(target)

summary = (
    out.groupBy("PLAN_SOURCE", "SCENARIO", "PLAN_VERSION")
    .agg(
        F.count("*").alias("ROW_COUNT"),
        F.sum("PLAN_AMOUNT_LOCAL").alias("TOTAL_PLAN_AMOUNT_LOCAL"),
    )
    .withColumn("GOLD_BATCH_ID", F.lit(BATCH_ID))
    .withColumn("GOLD_LOAD_TIMESTAMP", F.current_timestamp())
)
summary.write.mode("append").format("delta").saveAsTable(
    f"{AUD}.budget_fact_reconciliation"
)
display(summary)